## 1 概述

逻辑回归和 Softmax 回归的共同点是：通过梯度下降迭代优化一个损失函数来找到决策边界，本质上是一种“回归 → 分类”的转化思路。但分类问题不都要靠“回归”来解。有一类算法走的是另一条路：直接基于概率计算，不需要迭代。

朴素贝叶斯（ Naive Bayes, 简称 NB ）就是最经典的代表。它的核心思想非常简单：根据已有的统计数据，直接算出“新样本属于哪个类别的概率更大”。没有梯度，没有损失函数，只有贝叶斯定理和频数统计。

## 2 数据集

NB 算法要求所有特征和目标都是类别型，即离散型，不能是连续数值型，所以这里选择了汽车评估数据集，下载地址：https://archive.ics.uci.edu/dataset/19/car+evaluation

该数据集有样本 1728 个，输入特征 6 个，输出目标 1 个。

| 属性名 | 取值 | 含义 | 数据类型 |
| :--- | :--- | :--- | :--- |
| **buying** | v-high, high, med, low | 购买价格 | 类别型 |
| **maint** | v-high, high, med, low | 维护价格 | 类别型 |
| **doors** | 2, 3, 4, 5-more | 车门数量 | 类别型 |
| **persons** | 2, 4, more | 载客容量 | 类别型 |
| **lug_boot** | small, med, big | 行李箱大小 | 类别型 |
| **safety** | low, med, high | 安全性估计 | 类别型 |
| **class**（目标） | unacc, acc, good, v-good | 汽车可接受性 | 类别型 |

&nbsp;

| 类别 | 数量 | 占比 |
| :--- | ---: | ---: |
| **unacc**（不可接受） | 1210 | 70.023% |
| **acc**（可接受） | 384 | 22.222% |
| **good**（好） | 69 | 3.993% |
| **v-good**（非常好） | 65 | 3.762% |
| **合计** | **1728** | **100%** |

NB 算法要求特征的类别必须是整数（目标无此要求），但通常原始数据中类别都是字符串，所以需要转成整数索引，比如：
- buying 的 v-high, high, med, low 转成 0, 1, 2, 3
- safety 的 low, med, high 转成 0, 1, 2

所以数据集的预处理如下：


In [20]:
import pandas as pd
import json
import os

# 读取原始数据
names = ['buying','maint','doors','persons','lug_boot','safety','class']
df = pd.read_csv('./datasets/car/car.data', header=None, names=names)

X = df.drop('class', axis=1)    # 输入特征
y = df['class']                 # 输出目标

X_encoded = pd.DataFrame()
mappings = {}
for col in X.columns:
    cat_col = X[col].astype('category') # 将一列数据中出现的类别索引化
    X_encoded[col] = cat_col.cat.codes  # 索引化的内容代替原本的列
    # # 保存“索引→类别”的映射
    mappings[col] = {idx: val for idx, val in enumerate(cat_col.cat.categories)}
# 合并特征和目标
df_processed = X_encoded.copy()
df_processed['class'] = y
# 保存数据和映射
out_dir = "./outputs/nb"
os.makedirs(out_dir, exist_ok=True)
df_processed.to_csv(f'{out_dir}/car_processed.csv', index=False)
with open(f'{out_dir}/feature_mappings.json', 'w') as f:
    json.dump(mappings, f, indent=4)
# 打乱顺序，然后划分训练集和测试集
df_shuffled = df_processed.sample(frac=1, random_state=42).reset_index(drop=True)
# 计算切分位置
split_idx = int(len(df_shuffled) * 0.8)
# 按位置切分
df_train = df_shuffled.iloc[:split_idx]
df_test = df_shuffled.iloc[split_idx:]
# 保存切分好的训练集和测试集
df_train.to_csv(f'{out_dir}/car_train.csv', index=False)
df_test.to_csv(f'{out_dir}/car_test.csv', index=False)


## 3 贝叶斯定理

车的评级（即：类别 class ）用 C 表示，现在有辆车，其特征是 X ，那么它属于某个评级的概率是多少呢？贝叶斯定理告诉我们应该这么算：
$$
P(C|X)=\frac{P(X|C)\cdot P(C)}{P(X)}
$$

其中：
- $P(C)$ ：先验概率。指某个类别的占比，在训练集中计算：$P(C)=\frac{类别的数量}{总数}$ 。
- $P(C|X)$ ：后验概率。真正要求的目标，特征为 X 的这辆车属于某个类别的概率。
- $P(X|C)$ ：似然概率。在某个类别里，出现特征为 X 的概率。
- $P(X)$ ：特征 X 在所有车中出现的概率。

因为我们是要找到特征 X 属于哪个类别的概率最大，而作为分母的 $P(X)$ 是一样的，不影响对比的结果，所以其实只需要考虑分子即可，当然分子就不是概率了，可以认为是得分，得分高的就是概率大的：
$$
\text{Score}(C)=P(X|C)\cdot P(C)
$$

> 通常用小写表示单个，大写表示组合，而这里的 $C$ 其实是表示某个类别，有点别扭，但大部分资料都这么写。  
> 个人认为应该用 $c$ 或 $c_i$ 表示某个类别，$C$ 表示全部类别组成的向量，即 $C=[c_1,c_2,...,c_k]$ 或 $c\in C$ ，下文按此规则。

$P(c)$ 很好算，问题在于 $P(X|c)$ ，已知 X 是一个特征组合 $[x_1,\,x_2,\,...,x_m]$ ，所以：
$$
P(X|c)=P(x_1,\,x_2,\,...,\,x_m|c)
$$

## 4 朴素假设

接着计算就需要“朴素假设”：假设所有特征在给定类别 c 的条件下是相互独立的。在这个假设下，上式就可以拆开：
$$
P(X|c)=P(x_1,\,x_2,\,...,\,x_m|c)=P(x_1|c)\cdot P(x_2|c)\cdot ... \cdot P(x_m|c)
$$

那么算出所有类别的得分，取最大值，就应该是：
$$
\hat{y}=\arg\max_{c\in C}\left[P(c)\cdot\prod_{i=1}^mP(x_i|c)\right]
$$

> NB 版的“假设函数”

如此只需要计算 $P(x_i|c)$ 即可，很简单：$P(x_i|c)=\frac{\text{类别}\,c\,\text{中特征为}\,x_i\,\text{的数量}}{\text{类别}\,c\,\text{的总数}}=\frac{\text{count}(x_i,c)}{\text{count}(c)}$

## 5 拉普拉斯平滑

如果训练集中某个类别 (good) 的某个特征 (buying) 的取值没有出现过 (high) ，而到预测时来了一辆这样组合的新车（其 buying=high ），那么它的 $P(\text{buying}|\text{good})=0$ ，此时用上述公式计算：
$$
\hat{y}_{\text{good}}=P(X|\text{good})\cdot P(\text{good})=P(x_1|\text{good})\cdot ...\cdot P(\text{buying}|\text{good}) \cdot ... \cdot P(x_m|\text{good}) \cdot P(\text{good}) = 0
$$

意味着，因为训练集中没出现过价格高的车被评定为好车，那待评估的车只要是价格高，无论其他特征多么符合好车的条件，模型也不会将它判为好车（概率为零），这显然不合理，即使没见过这样的样本，它的概率也不该是零，只能说概率很低。

因此，为避免这种情况，给 $P(x_i|c)$ 的分子加个 1 ，确保任何特征、类别的组合至少出现过一次，分母加上该特征不同取值的个数 N ，即：$P(x_i|c)=\frac{\text{count}(x_i,c)+1}{\text{count}(c)+N_i}$

先验概率 $P(c)$ 通常也要进行拉普拉斯平滑，分子加 1 ，分母加上类别的数量 K ，即：$P(c)=\frac{\text{类别}\,c\,\text{的样本数量}+1}{\text{样本总数}+K}$

## 6 手写代码




In [ ]:
import pandas as pd
import numpy as np

# ========== 手写朴素贝叶斯分类器 ==========
class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha              # 拉普拉斯平滑参数，默认 1.0（即加一平滑）
        self.classes_ = None            # 所有类别列表
        self.class_prior_ = {}          # P(c)：每个类别的先验概率（对数）
        self.feature_probs_ = {}        # P(x_i|c)：条件概率表
        self.feature_n_values_ = {}     # 每个特征有多少种不同取值（用于平滑分母）
        self.class_counts_ = {}         # 用于处理未见过的值

    def fit(self, X, y):
        X = np.array(X)     # 转换为 numpy 数组方便处理
        y = np.array(y)
        n_samples, n_features = X.shape     # 样本数和特征数
        self.classes_ = np.unique(y)        # 类别列表
        n_classes = len(self.classes_)      # 类别数
        # 统计各类别计数
        self.class_counts_ = {c: np.sum(y == c) for c in self.classes_}
        # 先验概率 P(c) ，带拉普拉斯平滑
        for c in self.classes_:
            self.class_prior_[c] = (self.class_counts_[c] + self.alpha) / (n_samples + self.alpha * n_classes)
        # 记录每个特征有多少种取值（用于条件概率中分母进行拉普拉斯平滑，公式中的 N_i ）
        for j in range(n_features):
            self.feature_n_values_[j] = len(np.unique(X[:, j]))
        # 统计条件概率
        for c in self.classes_:
            mask = (y == c)
            X_c = X[mask]
            n_c = X_c.shape[0]
            self.feature_probs_[c] = {}
            for j in range(n_features):
                value_counts = {}
                for v in np.unique(X[:, j]):
                    value_counts[v] = np.sum(X_c[:, j] == v)
                self.feature_probs_[c][j] = {}
                for v, cnt in value_counts.items():
                    self.feature_probs_[c][j][v] = (cnt + self.alpha) / (n_c + self.alpha * self.feature_n_values_[j])

    # ========== 唯一的一个 _get_prob ==========
    def _get_prob(self, c, j, value):
        """安全查表：见过则返回概率，没见过则用平滑公式算一个极小概率"""
        if value in self.feature_probs_[c][j]:
            return self.feature_probs_[c][j][value]
        else:
            n_c = self.class_counts_[c]
            return self.alpha / (n_c + self.alpha * self.feature_n_values_[j])

    def predict_proba(self, X):
        """
        预测每个样本属于每个类别的概率（对数概率）
        返回：log_probs 矩阵，形状 (n_samples, n_classes)
        """
        X = np.array(X)
        log_probs = np.zeros((X.shape[0], len(self.classes_)))
        for i, sample in enumerate(X):
            for idx, c in enumerate(self.classes_):
                log_prob = np.log(self.class_prior_[c])
                for j, value in enumerate(sample):
                    log_prob += np.log(self._get_prob(c, j, value))
                log_probs[i, idx] = log_prob
        return log_probs

    def predict(self, X):
        """预测类别"""
        log_probs = self.predict_proba(X)
        return self.classes_[np.argmax(log_probs, axis=1)]

# ========== 加载数据 ==========
features = ['buying','maint','doors','persons','lug_boot','safety']
df_train = pd.read_csv('./outputs/nb/car_train.csv')
X_train = df_train[features]
y_train = df_train['class']
df_test = pd.read_csv('./outputs/nb/car_test.csv')
X_test = df_test[features]
y_test = df_test['class']
print(f"训练集: {len(X_train)} 样本, 测试集: {len(X_test)} 样本")
# ========== 训练模型 ==========
nb = NaiveBayes(alpha=1.0)
nb.fit(X_train, y_train)
# ========== 预测并评估 ==========
y_pred = nb.predict(X_test)
accuracy = np.mean(y_pred == y_test)
print(f"\n手写朴素贝叶斯准确率: {accuracy:.4f}")
# ========== 输出混淆矩阵 ==========
classes = nb.classes_
conf_mat = np.zeros((len(classes), len(classes)), dtype=int)

for true_label, pred_label in zip(y_test, y_pred):
    i = np.where(classes == true_label)[0][0]
    j = np.where(classes == pred_label)[0][0]
    conf_mat[i, j] += 1

print("\n混淆矩阵 (行=真实, 列=预测):")
print("        " + "  ".join(classes))
for i, c in enumerate(classes):
    row_str = f"{c:>8} " + "  ".join([str(conf_mat[i, j]) for j in range(len(classes))])
    print(row_str)

训练集: 1382 样本, 测试集: 346 样本

手写朴素贝叶斯准确率: 0.8208

混淆矩阵 (行=真实, 列=预测):
        acc  good  unacc  vgood
     acc 60  1  23  0
    good 17  3  0  0
   unacc 10  1  216  0
   vgood 10  0  0  5
